In [ ]:
import pandas as pd
import pulp
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus
import math

print("--- 正在为 '分形中心 F1' (f=4) 运行优化模型 ---")
print("--- 策略：将问题分解为4个独立的子模型 ---")

# --- 1. 基础数据定义 (Inputs) ---

# 一个分形中心 (f=4) 每周所需处理的分钟数
process_requirements = {
    'A': 17102.50, 'B': 54400.38, 'C': 13942.63, 'D': 77073.88,
    'E': 34105.56, 'F': 40754.94, 'G': 14080.69,
    'H': 52447.25, 'I': 47872.63, 'J': 80399.00,
    'K': 10809.19, 'L': 52733.63, 'M': 84480.50
}

# 每台机器每周的有效工作分钟数
EFFECTIVE_MINS_PER_WEEK = 4233.60

# --- 将设备数据分解为 4 个独立的组 ---

machine_data_groups = {
    "AD_Group": {
        "A": {"cost": 150000, "processes": ["A"]},
        "B": {"cost": 200000, "processes": ["B"]},
        "C": {"cost": 250000, "processes": ["C"]},
        "D": {"cost": 300000, "processes": ["D"]},
        "AB": {"cost": 250000, "processes": ["A", "B"]},
        "AC": {"cost": 280000, "processes": ["A", "C"]},
        "CD": {"cost": 350000, "processes": ["C", "D"]},
        "ABC": {"cost": 300000, "processes": ["A", "B", "C"]},
        "ABCD": {"cost": 400000, "processes": ["A", "B", "C", "D"]},
    },
    "EG_Group": {
        "E": {"cost": 400000, "processes": ["E"]},
        "F": {"cost": 400000, "processes": ["F"]},
        "G": {"cost": 400000, "processes": ["G"]},
        "EF": {"cost": 450000, "processes": ["E", "F"]},
        "EG": {"cost": 450000, "processes": ["E", "G"]},
        "FG": {"cost": 450000, "processes": ["F", "G"]},
        "EFG": {"cost": 500000, "processes": ["E", "F", "G"]},
    },
    "HJ_Group": {
        "H": {"cost": 1000000, "processes": ["H"]},
        "I": {"cost": 500000, "processes": ["I"]}, 
        "J": {"cost": 500000, "processes": ["J"]},
        "IJ": {"cost": 750000, "processes": ["I", "J"]}, 
    },
    "KM_Group": {
        "K": {"cost": 80000, "processes": ["K"]},
        "L": {"cost": 100000, "processes": ["L"]},
        "M": {"cost": 50000, "processes": ["M"]},
        "KL": {"cost": 140000, "processes": ["K", "L"]},
        "KM": {"cost": 100000, "processes": ["K", "M"]},
        "LM": {"cost": 120000, "processes": ["L", "M"]},
        "KLM": {"cost": 150000, "processes": ["K", "L", "M"]}
    }
}

# 对应的工艺分组
process_groups = {
    "AD_Group": ['A', 'B', 'C', 'D'],
    "EG_Group": ['E', 'F', 'G'],
    "HJ_Group": ['H', 'I', 'J'],
    "KM_Group": ['K', 'L', 'M']
}

# --- 2. 定义求解器函数 (Reusable Solver Function) ---

def solve_subproblem(group_name, group_processes, group_machines):
    """
    为单个独立的设备组求解最优机器组合
    """
    print(f"\n--- 正在求解子模型: {group_name} ---")
    
    prob = LpProblem(f"Fractal_F1_{group_name}", LpMinimize)
    MACHINE_NAMES = list(group_machines.keys())
    
    # --- 优化策略：计算并设置整数变量的下界 (Tighter Bounds) ---
    machine_low_bounds = {}
    for m in MACHINE_NAMES:
        # 仅为单一功能机器设置下界
        if len(group_machines[m]["processes"]) == 1:
            p = group_machines[m]["processes"][0]
            # 计算至少需要多少台该机器来满足需求
            min_required = math.ceil(process_requirements[p] / EFFECTIVE_MINS_PER_WEEK)
            machine_low_bounds[m] = min_required
            # print(f"  > (优化) 设置 {m} 的 LowBound = {min_required}") # (调试时取消注释)
        else:
            machine_low_bounds[m] = 0

    # 变量1 (N_m): 购买数量 (应用了优化的 lowBound)
    n_machines = LpVariable.dicts(
        "N",
        MACHINE_NAMES,
        lowBound=0, # PuLP 似乎不支持在 dicts 中为每个变量设置不同的 lowBound，我们在循环外单独设置
        cat='Integer'
    )
    
    # (在 PuLP 中，我们这样为变量设置单独的下界)
    for m, bound in machine_low_bounds.items():
        if bound > 0:
            n_machines[m].lowBound = bound

    # 变量2 (T_m,p): 时间分配
    time_allocations = LpVariable.dicts(
        "T",
        [(m, p) for m in MACHINE_NAMES for p in group_machines[m]["processes"]],
        lowBound=0,
        cat='Continuous'
    )

    # 目标函数: 最小化该组的成本
    prob += lpSum([
        group_machines[m]["cost"] * n_machines[m] for m in MACHINE_NAMES
    ]), f"Total_Cost_{group_name}"

    # 约束 1: 满足该组的工艺需求
    for p in group_processes:
        prob += lpSum([
            time_allocations[(m, p)] for m in MACHINE_NAMES
            if p in group_machines[m]["processes"]
        ]) >= process_requirements[p], f"Meet_Demand_for_{p}"

    # 约束 2: 该组的设备产能限制
    for m in MACHINE_NAMES:
        prob += lpSum([
            time_allocations[(m, p)] for p in group_machines[m]["processes"]
        ]) <= EFFECTIVE_MINS_PER_WEEK * n_machines[m], f"Capacity_Limit_for_{m}"

    # 求解
    solver_to_use = pulp.PULP_CBC_CMD(msg=False) # msg=False 关闭日志
    prob.solve(solver_to_use)

    # --- 收集结果 ---
    if LpStatus[prob.status] == 'Optimal':
        print(f"  > 状态: Optimal")
        
        group_cost = pulp.value(prob.objective)
        print(f"  > {group_name} 成本: ${group_cost:,.2f}")
        
        group_machine_mix = {}
        group_total_machines = 0
        
        print("  > 设备组合:")
        for m in MACHINE_NAMES:
            if n_machines[m].varValue is not None and n_machines[m].varValue > 0:
                count = int(round(n_machines[m].varValue))
                if count > 0:
                    print(f"    {m}: {count} 台")
                    group_machine_mix[m] = count
                    group_total_machines += count
        
        # (可选) 打印工时分配
        group_time_allocations = {}
        for (m, p) in time_allocations:
            if time_allocations[(m, p)].varValue is not None and time_allocations[(m, p)].varValue > 0.1:
                key = f"{m}_to_{p}"
                group_time_allocations[key] = time_allocations[(m, p)].varValue

        return {
            "name": group_name,
            "cost": group_cost,
            "machine_mix": group_machine_mix,
            "total_machines": group_total_machines,
            "time_allocations": group_time_allocations
        }
    else:
        print(f"  > 状态: {LpStatus[prob.status]} - 未找到最优解!")
        return None

# --- 3. 运行所有子模型 ---

results = []
for group_name, processes in process_groups.items():
    machines = machine_data_groups[group_name]
    result = solve_subproblem(group_name, processes, machines)
    if result:
        results.append(result)

# --- 4. 汇总总结果 ---
if len(results) == 4:
    print("\n" + "="*50)
    print("--- '分形中心 F1' 的最终最优设备方案 (Task 3f) ---")
    print("="*50)

    # 汇总成本和机器
    total_cost_f1 = sum(r['cost'] for r in results)
    total_machines_f1 = sum(r['total_machines'] for r in results)
    
    print(f"\n单个分形中心的最低设备投资 (F1): ${total_cost_f1:,.2f}")
    
    total_factory_cost = total_cost_f1 * 4 # f=4
    print(f"工厂总设备投资 (F1, F2, F3, F4): ${total_factory_cost:,.2f}")

    print("\n--- F1 中心的最优设备组合 (Machine_Mix) ---")
    full_machine_mix = {}
    for r in results:
        full_machine_mix.update(r['machine_mix'])
        
    for m, count in full_machine_mix.items():
        print(f"  {m}: {count} 台")
        
    print(f"  ---------------------")
    print(f"  F1 中心总机器数: {total_machines_f1} 台")
    print(f"  工厂总机器数 (f=4): {total_machines_f1 * 4} 台")

    print("\n--- F1 中心的工时分配 (Time_Allocations) ---")
    for r in results:
        print(f"  --- {r['name']} 分配 ---")
        for key, time in r['time_allocations'].items():
            print(f"    {key.replace('_to_', ' -> ')}: {time:,.1f} 分钟")

else:
    print("\n--- 优化失败，未能汇总所有子模型的结果。 ---")

print("\n--- 优化运行完毕 ---")

--- 正在为 '分形中心 F1' (f=4) 运行优化模型 ---
--- 策略：将问题分解为4个独立的子模型 ---

--- 正在求解子模型: AD_Group ---
  > 状态: Optimal
  > AD_Group 成本: $10,050,000.00
  > 设备组合:
    A: 5 台
    B: 13 台
    C: 4 台
    D: 19 台

--- 正在求解子模型: EG_Group ---
  > 状态: Optimal
  > EG_Group 成本: $9,200,000.00
  > 设备组合:
    E: 9 台
    F: 10 台
    G: 4 台

--- 正在求解子模型: HJ_Group ---
  > 状态: Optimal
  > HJ_Group 成本: $28,500,000.00
  > 设备组合:
    H: 13 台
    I: 12 台
    J: 19 台

--- 正在求解子模型: KM_Group ---
  > 状态: Optimal
  > KM_Group 成本: $2,540,000.00
  > 设备组合:
    K: 3 台
    L: 13 台
    M: 20 台

--- '分形中心 F1' 的最终最优设备方案 (Task 3f) ---

单个分形中心的最低设备投资 (F1): $50,290,000.00
工厂总设备投资 (F1, F2, F3, F4): $201,160,000.00

--- F1 中心的最优设备组合 (Machine_Mix) ---
  A: 5 台
  B: 13 台
  C: 4 台
  D: 19 台
  E: 9 台
  F: 10 台
  G: 4 台
  H: 13 台
  I: 12 台
  J: 19 台
  K: 3 台
  L: 13 台
  M: 20 台
  ---------------------
  F1 中心总机器数: 144 台
  工厂总机器数 (f=4): 576 台

--- F1 中心的工时分配 (Time_Allocations) ---
  --- AD_Group 分配 ---
    A -> A: 21,168.0 分钟
    B -> B: 55,036.8 分